# Eval TrafficCAM on india-yolov8n-final (CPU-friendly)

Tests the **detection half** and **queue half** of the screenshot-flow on real Indian CCTV footage.

| | |
|---|---|---
| Dataset | TrafficCAM (8 Indian cities, stationary cameras, open Google-Drive download — cite Deng et al., T-ITS 2024) |
| GPU needed | **No.** CPU runtime is fine. T4 optional speedup only. |
| Converter | `prepare_dataset.py --source trafficcam` (mask/polygon/RLE/box JSON → 6-class YOLO contract) |
| Hand-back | `trafficcam_eval_<date>.zip` → drop in `notebooks/training_output_zips/` |

Run cells top to bottom. Cell 2 (FirstFrame breadth) is optional — Cell 1 alone proves the pipeline.

In [ ]:
# Cell 0 — setup (CPU-only deps, ~3 min)
!pip install -q ultralytics onnxruntime opencv-python-headless gdown
!apt-get install -y -qq git-lfs > /dev/null 2>&1
!git lfs install --skip-repo > /dev/null 2>&1

import os, sys, shutil
REPO = '/content/SGP-IV'
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Bobbymkr/SGP-IV.git /content/SGP-IV
%cd /content/SGP-IV
!git lfs pull --include 'models/registry/india-yolov8n-final/*'
assert os.path.exists('models/registry/india-yolov8n-final/model-int8.onnx'), 'int8 model missing'
print('setup ok')

In [ ]:
# Cell 1 — download (cwd-proof: every `!` line is absolute; Colab resets
# the shell cwd on each line, so relative cd/gdown/unzip silently misplaces files)
!mkdir -p /content/trafficcam
!test -f /content/trafficcam/Fully_annotate.zip || gdown https://drive.google.com/uc?id=1h4oUqECDF05vSYMkYgz0aUc_RRzHlh3e -O /content/trafficcam/Fully_annotate.zip
!test -f /content/trafficcam/splits.zip || gdown https://drive.google.com/uc?id=1GcQ3J56kU-6TwRqRjzvI28-OF6QUdHeR -O /content/trafficcam/splits.zip
# rescue: an older revision of this cell dropped zips in /content — reuse them
!test -f /content/Fully_annotate.zip && cp -n /content/Fully_annotate.zip /content/trafficcam/ ; true
!test -f /content/splits.zip && cp -n /content/splits.zip /content/trafficcam/ ; true
!unzip -q -o /content/trafficcam/Fully_annotate.zip -d /content/trafficcam
!unzip -q -o /content/trafficcam/splits.zip -d /content/trafficcam
import glob
_n = len(glob.glob('/content/trafficcam/**/frame0.json', recursive=True))
assert _n >= 78, f'STOP: expected >=78 videos under /content/trafficcam, found {_n}'
print(f'download ok: {_n} videos')


In [ ]:
# Cell 2 — OPTIONAL breadth: FirstFrame partitions (2024 videos, frame0 labels only)
# Skip on slow connections; Cell 1 alone is sufficient for the verdict.
RUN_BREADTH = False
if RUN_BREADTH:
    !cd /content/trafficcam
    !gdown --id 1lYz_aBX-4Ygj-tbUkHvsaPB3vxOIvGBA -O FF1.zip
    !gdown --id 1TmN07tdC5NYpcIPI5tmhOrU1XfK7Yu6V -O FF2.zip
    !gdown --id 1cKOpDQ-NvqTxIETXYcDOv_Ojc3UZm2Qq -O FF3.zip
    !unzip -q -o FF1.zip && unzip -q -o FF2.zip && unzip -q -o FF3.zip
    print('breadth downloaded')
else:
    print('breadth skipped')

In [ ]:
# Cell 3 — convert to contract + validate (CPU, ~10-15 min for 64k frames if breadth on)
!cd /content/SGP-IV && python scripts/prepare_dataset.py /content/trafficcam \
    --out /content/tcam --source trafficcam --option B
!cd /content/SGP-IV && python scripts/prepare_dataset.py /content/tcam --make-calibration
!cd /content/SGP-IV && python scripts/prepare_dataset.py /content/tcam --check  # must print OKimport os
assert os.path.exists('/content/tcam/data.yaml'), \
    'STOP: converter wrote no data.yaml — read its output above for the layout error'


In [ ]:
# Cell 4 — bench: detection quality + latency + queue error (CPU)
# 4a. mAP of the canonical int8 model on TrafficCAM val (ultralytics validates ONNX directly)
!yolo val model=/content/SGP-IV/models/registry/india-yolov8n-final/model-int8.onnx \
    data=/content/tcam/data.yaml split=val 2>&1 | tail -15
# 4b. latency + per-frame histogram on disk frames
!cd /content/SGP-IV && python scripts/bench_detect.py --backend onnx \
    --registry models/registry/india-yolov8n-final --frames-dir /content/tcam/images/val --max-frames 200
# 4c. queue-vs-labels proxy (labels count ALL visible vs queue-zone: expect undercount bias;
#     the value is the per-weather gradient; honest qerr needs --gt-csv hand counts)
!cd /content/SGP-IV && python scripts/score_queue.py --frames-dir /content/tcam/images/val \
    --labels-dir /content/tcam/labels/val --backend onnx \
    --registry models/registry/india-yolov8n-final --conf 0.45 \
    --out-csv /content/tcam_queue_045.csv

In [ ]:
# Cell 5 — package hand-back zip (metrics + per-frame rows + data.yaml)
import datetime, glob
stamp = datetime.date.today().isoformat()
out = f'/content/trafficcam_eval_{stamp}.zip'
!cd /content && zip -qr {out} tcam_queue_045.csv tcam/data.yaml tcam/meta/captures.json
!ls -la {out}
from google.colab import files
files.download(out)
print('HAND-BACK:', out)
print('Paste into the issue: mAP50 overall + per-class, bench fps, queue RMSE/MAE/bias, detector-blind? (yes/no)')